# 03 — GRPO Reinforcement Learning for Translation Quality

**What this notebook does and why it exists**

After SFT, the model knows *how* to translate but it has not been explicitly optimised for *quality*. Supervised fine-tuning teaches the model to mimic the training data distribution; it does not teach it to prefer natural, idiomatic English over technically accurate but awkward phrasings.

This notebook applies Group Relative Policy Optimisation (GRPO) — a reinforcement learning algorithm — to push the model towards higher-quality outputs as judged by a large language model acting as a judge. The LLM judge evaluates each candidate translation for both *adequacy* (did it preserve the meaning?) and *fluency* (does it sound like natural spoken English?), and returns a combined score. The RL algorithm uses these scores to update the model weights in favour of translations that score higher.

**Why GRPO rather than PPO?** GRPO eliminates the separate value model required by PPO. Instead, for each input, it generates G candidate translations, scores them all, and uses the group mean and standard deviation to compute relative advantages. This makes GRPO far more memory-efficient — critical for a 1.5B model on Colab.

---
## What GRPO is doing that SFT cannot

SFT trains the model to maximise the probability of a specific reference translation. If the reference says *"And I think that is really the core of the issue"*, SFT rewards exactly those word choices. But there are dozens of equally good ways to translate a German sentence, and SFT penalises all of them.

GRPO works differently. For each German input:
1. The model generates G translations (default: 4)
2. Each translation is scored by the LLM judge
3. The scores are normalised relative to the group mean
4. The model is updated to increase the probability of above-average translations and decrease the probability of below-average ones

This means the model is rewarded for *any* good translation, not just the specific reference. It also learns what makes a translation *bad* by observing low-scoring candidates.

## Why an LLM judge rather than COMET as the RL reward?

COMET requires a reference translation. During RL, we want the reward to be reference-free — otherwise we are just doing a more expensive version of SFT. COMET-QE (quality estimation without reference) exists, but:
- Its scores are less interpretable
- It is hard to update as our quality criteria evolve
- It does not capture the *conversational register* quality we care about

An LLM judge, by contrast, can be prompted to evaluate exactly the dimensions we care about: does this sound like something a real English speaker would say in conversation? An LLM also produces richer variation in scores (which provides a better training signal) than a regression model that compresses everything to [0, 1].

**The risk:** LLM judges can be inconsistent. They may reward superficial features (e.g., longer outputs, formal register) rather than genuine quality. We guard against this by logging scores over time and inspecting the model's outputs.

---
## What the KL penalty is protecting against

GRPO (and RL fine-tuning in general) can cause the model to drift far from its pre-trained distribution. Without any constraint, the model could discover that a very short output like *"Good."* or a repetitive phrase like *"The the the the the"* receives a mediocre reward rather than a very low reward — and if the judge is slightly inconsistent, the model might exploit this.

The KL (Kullback-Leibler) divergence penalty adds a term to the loss that penalises the policy model for diverging too far from the SFT reference model. It acts like a leash: the model can move towards higher-reward outputs, but not too far from where it started.

- **Too high a KL penalty (β > 0.5):** The model barely moves from SFT. RL has little effect.
- **Too low a KL penalty (β < 0.001):** The model is free to explore widely, which risks reward hacking and mode collapse.
- **Default (β = 0.04):** A reasonable starting point for translation. Adjust based on the reward score trajectory.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# Unsloth's GRPO implementation is the most memory-efficient available.
# Note: Unsloth requires a CUDA GPU. If you are on a TPU, use TRL's GRPOTrainer.
# We install both and select at runtime.
!pip install -q \
    trl==0.11.4 \
    peft==0.13.2 \
    transformers==4.45.0 \
    datasets==2.21.0 \
    accelerate==0.34.2 \
    openai==1.54.0 \
    torch==2.4.0

# For GPU runtime with Unsloth GRPO support (faster and lower VRAM):
# !pip install -q unsloth[colab-new] bitsandbytes>=0.43.0

print('Dependencies installed.')

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
BASE_DIR    = '/content/drive/MyDrive/podcast_translation'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
SFT_ADAPTER = os.path.join(BASE_DIR, 'sft_adapter')
GRPO_DIR    = os.path.join(BASE_DIR, 'grpo_adapter')
os.makedirs(GRPO_DIR, exist_ok=True)

# Load OpenRouter API key from Colab secrets (never hardcode keys)
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError(
        'OPENROUTER_API_KEY not found in Colab secrets. '
        'Go to the key icon in the left sidebar and add it.'
    )
print('API key loaded.')

In [ ]:
import json
import random
import time
import re
from typing import Optional

import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from openai import OpenAI

random.seed(42)
torch.manual_seed(42)

BASE_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN     = 256
print('Imports complete.')

In [ ]:
# ── OpenRouter client ─────────────────────────────────────────────────────────
# We use openai's SDK pointed at OpenRouter's OpenAI-compatible endpoint.
# Model: openai/gpt-5.2 — verified as a valid OpenRouter model ID (February 2026).
# If this model is unavailable or too expensive, fallback to openai/gpt-5 or
# google/gemini-2.0-flash-001 which is cheaper and still strong at evaluation.

JUDGE_MODEL = 'openai/gpt-5.2'   # Verified against openrouter.ai/models
# JUDGE_MODEL = 'openai/gpt-5'   # Cheaper fallback
# JUDGE_MODEL = 'google/gemini-2.0-flash-001'  # Cheapest reasonable fallback

openrouter_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)
print(f'OpenRouter client ready. Judge model: {JUDGE_MODEL}')

---
## Decision note: The reward prompt

The reward prompt is the most important design decision in this notebook. It defines what "good translation" means for the purpose of RL training. We ask the judge to evaluate two separate dimensions:

1. **Adequacy (1–10):** Did the translation preserve the meaning of the original? A score of 1 means a mistranslation; 10 means perfect semantic equivalence.
2. **Fluency (1–10):** Does the translation sound like natural spoken English? A score of 1 means awkward or ungrammatical; 10 means it reads as something a fluent speaker would naturally say in conversation.

We combine them into a single scalar via a weighted sum (70% adequacy, 30% fluency — weighted towards adequacy because a fluent but inaccurate translation is worse than an accurate but slightly awkward one).

**To tune the reward prompt:** Change the weights, the scale (1–5 instead of 1–10), or add a third dimension like *register* (is the English appropriately conversational?). After changing the prompt, inspect a batch of scores before resuming training — the mean score should be around 5–7 for SFT-trained outputs.

In [ ]:
# ── Reward function ───────────────────────────────────────────────────────────
# This string constant is intentionally visible so you can inspect and tune it.
REWARD_SYSTEM_PROMPT = """You are an expert translator and linguist evaluating German-to-English translations of podcast transcripts.

For each translation candidate you will receive:
  SOURCE: the original German sentence
  TRANSLATION: the proposed English translation

Evaluate the translation on exactly two dimensions:

1. ADEQUACY (1-10): Does the translation accurately convey the meaning of the German original?
   - 1 = completely wrong or missing meaning
   - 5 = roughly correct but with notable omissions or additions  
   - 10 = perfect semantic equivalence

2. FLUENCY (1-10): Does the translation sound like natural spoken English?
   - 1 = grammatically broken or unnatural
   - 5 = grammatically correct but stilted, like a textbook
   - 10 = sounds exactly like something a native English speaker would say naturally in conversation

Respond with ONLY a JSON object in this exact format, nothing else:
{"adequacy": <int>, "fluency": <int>}"""

ADEQUACY_WEIGHT = 0.70
FLUENCY_WEIGHT  = 0.30

def score_translation(
    german: str,
    english: str,
    retries: int = 3,
    backoff: float = 2.0
) -> Optional[float]:
    """Call the LLM judge and return a combined score in [0, 1].
    
    Returns None if the API call fails after all retries.
    A None reward is treated as the group mean (neutral signal).
    """
    user_message = f'SOURCE: {german}\nTRANSLATION: {english}'

    for attempt in range(retries):
        try:
            response = openrouter_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {'role': 'system', 'content': REWARD_SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_message},
                ],
                max_tokens=32,
                temperature=0.0,  # Deterministic scoring
            )
            raw = response.choices[0].message.content.strip()
            scores = json.loads(raw)
            adequacy = float(scores['adequacy'])
            fluency  = float(scores['fluency'])
            # Normalise to [0, 1]
            combined = (ADEQUACY_WEIGHT * adequacy + FLUENCY_WEIGHT * fluency) / 10.0
            return combined
        except Exception as e:
            wait = backoff ** attempt
            print(f'Reward API error (attempt {attempt+1}/{retries}): {e}. Retrying in {wait}s...')
            time.sleep(wait)
    return None

# Quick test
test_score = score_translation(
    'Das ist ein Test.',
    'This is a test.'
)
print(f'Test score: {test_score} (expect ~0.9)')

In [ ]:
# ── Load SFT checkpoint ───────────────────────────────────────────────────────
print(f'Loading base model: {BASE_MODEL_NAME}')
tokeniser = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokeniser.pad_token is None:
    tokeniser.pad_token = tokeniser.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

print(f'Loading SFT LoRA adapter from: {SFT_ADAPTER}')
model = PeftModel.from_pretrained(base_model, SFT_ADAPTER)
model = model.merge_and_unload()   # Merge for GRPO training — we add new adapter below
print('SFT checkpoint loaded and merged.')

In [ ]:
# ── Add new LoRA adapter for GRPO phase ───────────────────────────────────────
# We start fresh with a new adapter on top of the merged SFT model.
# This gives us a clean reference policy (the merged SFT model) and
# a clean policy model (the merged SFT + new LoRA adapter).
from peft import LoraConfig, get_peft_model, TaskType

grpo_lora = LoraConfig(
    r=8,              # Smaller rank for RL phase — we want fine-grained adjustments
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.0,  # No dropout during RL — we want stable gradient estimates
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, grpo_lora)
model.print_trainable_parameters()

In [ ]:
# ── Load training data ────────────────────────────────────────────────────────
def load_jsonl(path: str) -> list:
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
val_data   = load_jsonl(os.path.join(DATA_DIR, 'val.jsonl'))

PROMPT_TEMPLATE = (
    'Translate the following German podcast transcript to natural English.\n\n'
    'German: {german}\n\n'
    'Translation:'
)

# For GRPO we only need the prompts (inputs), not the references
grpo_prompts = [
    {'prompt': PROMPT_TEMPLATE.format(german=p['de']), 'german': p['de']}
    for p in train_data
]
random.shuffle(grpo_prompts)

print(f'Training prompts for GRPO: {len(grpo_prompts):,}')

---
## GRPO Hyperparameters — What They Control

| Parameter | Value | What it does |
|---|---|---|
| `num_generations` (G) | 4 | How many translation candidates to generate per input. More candidates → better advantage estimates → slower reward calls. 4 is a good balance. |
| `kl_coeff` (β) | 0.04 | Weight of the KL penalty. Higher = stays closer to SFT. Lower = more freedom but more risk of hacking. |
| `epsilon` | 0.2 | Reward clipping range (from PPO-style clipping). Prevents any single example from dominating the gradient. |
| `learning_rate` | 5e-6 | Much lower than SFT — RL is already making large changes via the reward signal. |
| `max_new_tokens` | 150 | Maximum translation length. Must be consistent with inference settings. |

**Failure modes to watch for:**
- **Reward hacking:** The model learns to produce outputs that score well on the judge prompt but are not actually good translations. Signs: scores rise quickly but qualitative quality seems flat or odd.
- **Mode collapse:** The model starts producing the same translation for all inputs. Signs: very low output diversity, repetitive phrases.
- **Length reward exploitation:** The model produces unusually long or short outputs. Monitor mean output length.

In [ ]:
# ── GRPO Training Loop ────────────────────────────────────────────────────────
# We implement a manual GRPO loop to give maximum visibility into what is
# happening. TRL's GRPOTrainer is an alternative but less transparent.
# This loop can also be adapted to use Unsloth's GRPOTrainer on GPU.

import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Configurable hyperparameters ──────────────────────────────────────────────
G               = 4          # Number of candidates per input
# Note: KL_COEFF and EPSILON are defined here for reference but are not
# applied in this simplified implementation (basic REINFORCE with normalised
# advantages). To add proper GRPO, implement KL penalty against ref policy
# and PPO-style ratio clipping using these values.
LR              = 5e-6
MAX_NEW_TOKENS  = 150
GRPO_STEPS      = 500        # Total update steps
SAVE_EVERY      = 100        # Save checkpoint every N steps
LOG_EVERY       = 10

optimiser   = AdamW(model.parameters(), lr=LR)
scheduler   = CosineAnnealingLR(optimiser, T_max=GRPO_STEPS)

# Monitoring buffers
score_history    = []   # Raw LLM judge scores over time
length_history   = []   # Mean output token length over time
step_data_index  = 0

# Note: this implementation uses REINFORCE with normalised advantages, not full GRPO.
# A reference model for KL penalty is not maintained here.
model.train()
device = next(model.parameters()).device

print('Starting GRPO training...')
for step in range(1, GRPO_STEPS + 1):
    # Sample a batch (1 example per step for clarity — extend to mini-batch if memory allows)
    if step_data_index >= len(grpo_prompts):
        step_data_index = 0
        random.shuffle(grpo_prompts)
    
    example = grpo_prompts[step_data_index]
    step_data_index += 1
    
    prompt_text  = example['prompt'] + ' '
    german_text  = example['german']
    
    prompt_ids = tokeniser(
        prompt_text,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).input_ids.to(device)
    
    # ── Step 1: Generate G candidate translations ──────────────────────────
    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            prompt_ids.repeat(G, 1),
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokeniser.eos_token_id
        )
    
    candidates = []
    for out in outputs:
        new_tokens = out[prompt_ids.shape[-1]:]
        text = tokeniser.decode(new_tokens, skip_special_tokens=True).strip()
        candidates.append(text)
    
    # ── Step 2: Score all candidates via LLM judge ─────────────────────────
    raw_scores_raw = [score_translation(german_text, c) for c in candidates]
    valid_scores = [s for s in raw_scores_raw if s is not None]
    group_mean = sum(valid_scores) / len(valid_scores) if valid_scores else 0.5
    raw_scores = [s if s is not None else group_mean for s in raw_scores_raw]
    
    # ── Reward hacking guards ──────────────────────────────────────────────
    output_lengths = [len(tokeniser.encode(c)) for c in candidates]
    mean_len = sum(output_lengths) / len(output_lengths)
    length_history.append({'step': step, 'mean_len': mean_len})
    score_history.append({'step': step, 'mean_score': sum(raw_scores)/len(raw_scores)})
    
    # Flag suspicious outputs
    if mean_len < 3:
        print(f'WARNING Step {step}: mean output length = {mean_len:.1f} — model may be collapsing')
    if len(set(candidates)) == 1:
        print(f'WARNING Step {step}: all G candidates are identical — mode collapse suspected')
    
    # ── Step 3: Compute GRPO advantages ───────────────────────────────────
    scores_t = torch.tensor(raw_scores, dtype=torch.float32)
    mean_score = scores_t.mean()
    std_score  = scores_t.std() + 1e-8
    advantages = (scores_t - mean_score) / std_score   # Normalised advantages
    
    # ── Step 4: Policy gradient update ───────────────────────────────────
    model.train()
    total_loss = torch.tensor(0.0, requires_grad=True, device=device)
    
    for i, (candidate, adv) in enumerate(zip(candidates, advantages)):
        full_text = prompt_text + candidate
        full_ids  = tokeniser(
            full_text,
            return_tensors='pt',
            truncation=True,
            max_length=MAX_SEQ_LEN + MAX_NEW_TOKENS
        ).input_ids.to(device)
        
        prompt_len = prompt_ids.shape[-1]
        
        logits = model(full_ids).logits
        # Compute log-prob of generated tokens only (not the prompt)
        gen_logits = logits[0, prompt_len-1:-1, :]
        gen_ids    = full_ids[0, prompt_len:]
        
        if gen_ids.numel() == 0:
            continue
        
        log_probs = F.log_softmax(gen_logits, dim=-1)
        token_log_probs = log_probs.gather(
            1, gen_ids.unsqueeze(-1)
        ).squeeze(-1)
        seq_log_prob = token_log_probs.mean()
        
        # Policy gradient loss (negated because we minimise)
        pg_loss = -adv.to(device) * seq_log_prob
        total_loss = total_loss + pg_loss / G
    
    optimiser.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimiser.step()
    scheduler.step()
    
    if step % LOG_EVERY == 0:
        mean_sc = sum(raw_scores) / len(raw_scores)
        print(f'Step {step:4d} | loss={total_loss.item():.4f} | '
              f'mean_score={mean_sc:.3f} | mean_len={mean_len:.1f}')
        print(f'  Best candidate: {candidates[raw_scores.index(max(raw_scores))][:100]}')
    
    if step % SAVE_EVERY == 0:
        ckpt_path = os.path.join(GRPO_DIR, f'checkpoint_{step}')
        model.save_pretrained(ckpt_path)
        print(f'Checkpoint saved → {ckpt_path}')

print('GRPO training complete.')

In [ ]:
# ── Save final GRPO adapter ───────────────────────────────────────────────────
# IMPORTANT: this adapter was trained on top of the *merged SFT weights*,
# not the raw base model. To apply it correctly at inference/eval time,
# you must first load and merge the SFT adapter, then apply this GRPO adapter.
final_path = os.path.join(GRPO_DIR, 'final')
model.save_pretrained(final_path)
tokeniser.save_pretrained(final_path)
print(f'Final GRPO adapter saved → {final_path}')

In [ ]:
# ── Plot RL monitoring metrics ────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

steps  = [x['step'] for x in score_history]
scores = [x['mean_score'] for x in score_history]
axes[0].plot(steps, scores, color='mediumseagreen')
axes[0].set_title('LLM judge score (mean over G candidates)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Score (0–1)')
axes[0].axhline(0.5, linestyle='--', color='grey', alpha=0.5)

lens = [x['mean_len'] for x in length_history]
axes[1].plot(steps, lens, color='steelblue')
axes[1].set_title('Mean output token length (reward hacking indicator)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Tokens')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'grpo_monitoring.png'), dpi=150)
plt.show()

print('\nINTERPRETING THESE PLOTS:')
print('  Score plot should rise gradually, not spike then plateau.')
print('  Length plot should be stable (~30–80 tokens). Sharp rise = verbose hacking.')
print('  Sharp score drop after initial rise = catastrophic forgetting; reduce KL_COEFF.')

---
## Failure Modes and How to Detect Them

**Reward hacking:** The model learns to exploit the judge's scoring prompt rather than producing genuinely good translations. Early warning signs:
- Scores rise to 0.9+ within the first 50 steps (too fast)
- Output lengths drift significantly (shorter = lazy; longer = padding)
- Qualitative inspection reveals repetitive or formulaic phrasing that "sounds" good but says little

**Mitigation:** Increase `KL_COEFF`; add more diverse judge prompts; switch judge models occasionally.

**Mode collapse:** The model starts producing the same translation for all inputs.
- Warning: the `set(candidates) == 1` check above fires repeatedly
- Mitigation: Reduce `KL_COEFF`; increase sampling temperature; reduce LoRA rank

**Catastrophic forgetting:** The model forgets how to translate German entirely and produces English text unrelated to the input.
- Warning: COMET scores on the validation set drop below the SFT baseline
- Mitigation: Restore from earlier checkpoint; increase `KL_COEFF`; reduce learning rate

**Proceed to: `04_evaluation.ipynb`**